# 05 - merged h5ad の検収 ＋ 探索的可視化

このノートブックは、本解析ではなく merged h5ad の検収と探索的可視化を目的とする。

```
04dで作成した merged_qc_original_scale は original-scale merge であり、.X には raw_count_like, cpm_tpm_like, log_normalized_like が混在している。

そのため、PCA/UMAP/Harmony/clusteringに使う前に make_logexpr_copy_from_original_scale() で探索用copyを作る。

この探索用copyは raw_count_like と cpm_tpm_like を normalize_total(target_sum=1e4) + log1p し、log_normalized_like はそのまま使う。

この探索用copyは可視化・クラスタリング確認用であり、scVIなどのcount-based modelの入力には使わない。

outer gene setは遺伝子の和集合、inner gene setは全datasetに共通する遺伝子集合である。

outerではdatasetに存在しなかった遺伝子が0埋めされる可能性があり、PCAにbatch由来の影響を与える可能性がある。

innerでは共通遺伝子だけを使うため比較しやすいが、重要なHVGや細胞種マーカーが落ちている可能性がある。

したがって、PCA/UMAP/Harmony/clusteringの前にouter/inner gene setの妥当性を確認する。

PanglaoDB参照マーカーに基づく自動アノテーションは provisional annotation であり、本番のcell type annotationではない。
必ず既知annotation、marker expression、cluster marker genesと照合して解釈する。
```

## このノートブックがやらないこと（制約）

- original-scale の `merged_qc_original_scale_outer.h5ad` / `inner.h5ad` を上書きしない。
- original-scale h5ad に直接 `normalize_total` / `log1p` / `scale` をかけて上書きしない（必ず copy に対して行う）。
- 自動アノテーションは provisional / exploratory であり、本番 annotation として扱わない。


## セットアップ（import・パス・scanpy 設定）

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

sc.settings.verbosity = 1

ROOT = Path("../../").resolve()

MERGED_DIR = ROOT / "data" / "merged_h5ad"
QC_H5AD_DIR = ROOT / "data" / "qc_h5ad"
SIDECAR_PATH = ROOT / "data" / "curated_h5ad" / "original_obs_metadata_by_cell.csv"

OUT_DIR = ROOT / "results" / "check_merged_h5ad"
PLOT_DIR = OUT_DIR / "plots"
GENESET_DIR = OUT_DIR / "gene_set"
PCA_UMAP_DIR = OUT_DIR / "pca_umap"
CLUSTER_DIR = OUT_DIR / "clustering"
ANNOTATION_DIR = OUT_DIR / "annotation"

for d in (OUT_DIR, PLOT_DIR, GENESET_DIR, PCA_UMAP_DIR, CLUSTER_DIR, ANNOTATION_DIR):
    d.mkdir(parents=True, exist_ok=True)

STATES = ["raw_count_like", "cpm_tpm_like", "log_normalized_like"]
print("ROOT:", ROOT)


## 共通ヘルパー関数

In [ ]:
def san(s):
    """ファイル名用に英数以外を _ に置換。"""
    return re.sub(r"[^0-9A-Za-z]+", "_", str(s)).strip("_")


def get_gene_symbols_upper(adata):
    if "gene_symbol_upper" in adata.var.columns:
        return adata.var["gene_symbol_upper"].astype(str).str.upper()
    if "gene_symbol" in adata.var.columns:
        return adata.var["gene_symbol"].astype(str).str.upper()
    return pd.Index(adata.var_names).astype(str).str.upper()


def normalize_gene_symbol(x):
    if pd.isna(x):
        return None
    return str(x).strip().upper()


def get_mt_mask_from_names(names):
    up = pd.Index([str(n).upper() for n in names])
    return np.asarray(up.str.startswith("MT-"))


def get_ribo_mask_from_names(names):
    up = pd.Index([str(n).upper() for n in names])
    return np.asarray(up.str.startswith(("RPS", "RPL", "MRPS", "MRPL")))


def savefig(path):
    plt.savefig(path, dpi=100, bbox_inches="tight")
    plt.close("all")


def emb_color(adata, basis, keys, prefix, outdir):
    """basis (例 'umap_before_harmony') 上で keys を色付けして保存。無い列はskip。"""
    for k in keys:
        present = (k in adata.obs.columns) or (k in adata.var_names)
        if not present:
            print(f"  [skip] color '{k}' not present")
            continue
        try:
            sc.pl.embedding(adata, basis=basis, color=k, show=False)
            savefig(outdir / f"{prefix}_{san(k)}.png")
        except Exception as e:
            print(f"  [warn] embedding {basis} color={k} failed: {e}")
            plt.close("all")


def stacked_fraction_plot(ct, title, path, legend_title=None, fontsize=6):
    frac = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)
    ax = frac.plot(kind="bar", stacked=True, figsize=(max(6, 0.4 * len(frac)), 4))
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.legend(title=legend_title, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=fontsize)
    plt.tight_layout()
    savefig(path)


## 1. merged h5ad を読み込み、基本チェック

In [ ]:
outer = sc.read_h5ad(MERGED_DIR / "merged_qc_original_scale_outer.h5ad")
inner = sc.read_h5ad(MERGED_DIR / "merged_qc_original_scale_inner.h5ad")

print("outer:", outer.shape)
print("inner:", inner.shape)

inner_genes = set(inner.var_names)
outer_only_genes = [g for g in outer.var_names if g not in inner_genes]

basic = {
    "outer_n_cells": outer.n_obs,
    "outer_n_genes": outer.n_vars,
    "inner_n_cells": inner.n_obs,
    "inner_n_genes": inner.n_vars,
    "outer_obs_names_unique": bool(outer.obs_names.is_unique),
    "inner_obs_names_unique": bool(inner.obs_names.is_unique),
    "outer_var_names_unique": bool(outer.var_names.is_unique),
    "inner_var_names_unique": bool(inner.var_names.is_unique),
    "n_cells_match": bool(outer.n_obs == inner.n_obs),
    "obs_names_match": bool(list(outer.obs_names) == list(inner.obs_names)),
    "n_outer_only_genes": len(outer_only_genes),
    "n_inner_genes": inner.n_vars,
}
basic_df = pd.DataFrame({"key": list(basic.keys()), "value": list(basic.values())})
basic_df.to_csv(OUT_DIR / "merged_basic_check.csv", index=False)
if not basic["obs_names_match"]:
    print("[WARN] outer/inner obs_names do not match in order/content")
basic_df


## 2. known annotation を sidecar から戻す

`original_obs_metadata_by_cell.csv` を `cell_uid` で join。存在しない列は warning のみ。

In [ ]:
try:
    sidecar = pd.read_csv(SIDECAR_PATH, low_memory=False)
    if "cell_uid" not in sidecar.columns:
        raise ValueError("sidecar has no cell_uid column")
    sidecar = sidecar.drop_duplicates("cell_uid").set_index("cell_uid")
    print("sidecar:", sidecar.shape)
except Exception as e:
    print(f"[WARN] could not load sidecar: {e}")
    sidecar = None


def restore_sidecar_columns(adata, sidecar, mapping):
    """mapping: {sidecar_col: target_obs_col}. 存在しない列は warning。"""
    if sidecar is None:
        print("  [warn] sidecar not available; skip restore")
        return
    key = adata.obs["cell_uid"].astype(str) if "cell_uid" in adata.obs.columns else pd.Index(adata.obs_names).astype(str)
    for src_col, tgt_col in mapping.items():
        if src_col not in sidecar.columns:
            print(f"  [warn] sidecar missing column: {src_col}")
            continue
        adata.obs[tgt_col] = key.map(sidecar[src_col]).values


# GSE173524 と GSE206330 の列を戻す（unified 列と衝突しない名前で）
RESTORE_MAP = {
    # GSE173524
    "meta_sex": "meta_sex",
    "meta_cell.type.refined": "meta_cell.type.refined",
    "meta_microglia.subclusters": "meta_microglia.subclusters",
    "meta_provisional.microglia.subclusters": "meta_provisional.microglia.subclusters",
    "meta_astrocyte.subclusters": "meta_astrocyte.subclusters",
    "meta_oligodendrocyte.subclusters": "meta_oligodendrocyte.subclusters",
    # GSE206330
    "Age": "Age",
    "sex": "sex_GSE206330",                       # 元 sex（後で unified sex に統合）
    "cell_type": "cell_type_original_GSE206330",  # 元 cell_type（統合列で上書きしない）
}

for adata in (outer, inner):
    restore_sidecar_columns(adata, sidecar, RESTORE_MAP)

[c for c in ["meta_sex", "meta_cell.type.refined", "Age", "sex_GSE206330", "cell_type_original_GSE206330"] if c in outer.obs.columns]


## 3. `sex` 列を統合

- GSE173524: `meta_sex` をそのまま
- GSE206330: `sex` を `{"Female":"F","Male":"M"}` に変換
- それ以外: `unknown`

In [ ]:
def build_unified_sex(adata):
    obs = adata.obs
    sex = pd.Series(np.nan, index=obs.index, dtype=object)
    if "meta_sex" in obs.columns:
        sex = sex.where(obs["meta_sex"].isna(), obs["meta_sex"].astype(object))
    if "sex_GSE206330" in obs.columns:
        s2 = obs["sex_GSE206330"].map({"Female": "F", "Male": "M"})
        sex = sex.where(s2.isna(), s2)
    adata.obs["sex"] = sex.fillna("unknown").astype(str)


for adata in (outer, inner):
    build_unified_sex(adata)

outer.obs["sex"].value_counts(dropna=False)


## 4. `cell_type` 列を統合

GSE173524 `meta_cell.type.refined` と GSE206330 `cell_type`（→ `cell_type_original_GSE206330`）を
rename して統合列 `cell_type` を作る。元列は残す。

In [ ]:
RENAME_173524_CT = {
    "Oligodendrocyte": "Oligodendrocyte",
    "Astrocyte": "Astrocyte",
    "Endothelial": "Endothelial",
    "Neuron": "Neuron",
    "Schwann_Meningeal": "Schwann_Meningeal",
    "Microglia": "Microglia",
    "Ependymal": "Schwann_Meningeal",
    "OPC": "OPC",
    "Fibroblast": "Fibroblast",
    "Motor_Neuron": "Motor_Neuron",
}
RENAME_206330_CT = {
    "Microglia": "Microglia",
    "Endothelial cells/Pericytes": "Endothelial cells/Pericytes",
    "Oligodendrocytes": "Oligodendrocyte",
    "Astrocytes": "Astrocyte",
    "Committed oligodendrocyte precursors": "OPC",
    "Perivascular macrophages": "Perivascular macrophages",
    "Pericytes": "Pericytes",
    "Endothelial cells": "Endothelial",
    "Oligodendrocyte precursors": "OPC",
}


def build_unified_cell_type(adata):
    obs = adata.obs
    ct = pd.Series(np.nan, index=obs.index, dtype=object)
    if "meta_cell.type.refined" in obs.columns:
        c1 = obs["meta_cell.type.refined"].map(RENAME_173524_CT)
        ct = ct.where(c1.isna(), c1)
    if "cell_type_original_GSE206330" in obs.columns:
        c2 = obs["cell_type_original_GSE206330"].map(RENAME_206330_CT)
        ct = ct.where(c2.isna(), c2)
    adata.obs["cell_type"] = ct.fillna("unknown").astype(str)


for adata in (outer, inner):
    build_unified_cell_type(adata)

outer.obs["cell_type"].value_counts(dropna=False).head(20)


## 5-6. dataset ごとの遺伝子存在確認

**重要**: outer 行列の 0 で「存在しない」と判定しない。各 `*.stage1_filtered.h5ad` の
`var_names`（`get_gene_symbols_upper` で outer と同じ正規化）を使って判定する。

In [ ]:
# 各 dataset の遺伝子集合（uppercase canonical）を stage1_filtered から取得
dataset_genes = {}     # dataset_id -> set(genes)
dataset_meta = {}      # dataset_id -> (source_accession, state)
for state in STATES:
    for p in sorted((QC_H5AD_DIR / state).glob("*.stage1_filtered.h5ad")):
        dataset_id = p.name.replace(".stage1_filtered.h5ad", "")
        a = sc.read_h5ad(p)
        genes = set(pd.Index(get_gene_symbols_upper(a)).astype(str))
        dataset_genes[dataset_id] = genes
        acc = str(a.obs["source_accession"].iloc[0]) if "source_accession" in a.obs.columns else dataset_id.split("_")[0]
        dataset_meta[dataset_id] = (acc, state)
        del a

dataset_ids = sorted(dataset_genes)
n_datasets = len(dataset_ids)
print(f"{n_datasets} datasets:", dataset_ids)

# gene x dataset presence (0/1)
all_genes = list(outer.var_names)
presence = pd.DataFrame(
    {did: pd.Index(all_genes).isin(dataset_genes[did]).astype(int) for did in dataset_ids},
    index=all_genes,
)
presence.index.name = "gene"
presence.to_csv(GENESET_DIR / "gene_presence_by_dataset.csv")

# nonzero stats from outer.var if available else compute
if "n_nonzero_cells" in outer.var.columns:
    nnz_outer = outer.var["n_nonzero_cells"].reindex(all_genes).to_numpy()
    pct_outer = outer.var["pct_nonzero_cells"].reindex(all_genes).to_numpy() if "pct_nonzero_cells" in outer.var.columns else 100.0 * nnz_outer / outer.n_obs
else:
    Xo = outer.X
    nnz_outer = Xo.getnnz(axis=0) if sp.issparse(Xo) else (np.asarray(Xo) != 0).sum(axis=0)
    nnz_outer = np.asarray(nnz_outer).ravel()
    pct_outer = 100.0 * nnz_outer / outer.n_obs

n_present = presence.sum(axis=1).to_numpy()
inner_gene_set = set(inner.var_names)
present_lists, missing_lists = [], []
pmat = presence.to_numpy().astype(bool)
ds_arr = np.array(dataset_ids)
for i in range(len(all_genes)):
    present_lists.append(";".join(ds_arr[pmat[i]]))
    missing_lists.append(";".join(ds_arr[~pmat[i]]))

gene_presence_summary = pd.DataFrame({
    "gene": all_genes,
    "n_datasets_present": n_present,
    "n_datasets_missing": n_datasets - n_present,
    "datasets_present": present_lists,
    "datasets_missing": missing_lists,
    "in_inner": [g in inner_gene_set for g in all_genes],
    "outer_only": [g not in inner_gene_set for g in all_genes],
    "n_nonzero_cells_outer": nnz_outer,
    "pct_nonzero_cells_outer": pct_outer,
})
gene_presence_summary.to_csv(GENESET_DIR / "gene_presence_summary.csv", index=False)
print("saved gene presence CSVs")
gene_presence_summary.head()


## 7. dataset ごとの保持遺伝子数

In [ ]:
rows = []
for did in dataset_ids:
    acc, state = dataset_meta[did]
    gset = dataset_genes[did]
    n_in = len(gset)
    overlap_inner = len(gset & inner_gene_set)
    unique_to = sum(1 for g in gset if n_datasets >= 1 and presence.loc[g].sum() == 1) if n_in else 0
    rows.append({
        "source_accession": acc,
        "dataset_id": did,
        "qc_preprocessing_state": state,
        "n_genes_in_dataset": n_in,
        "n_genes_overlap_inner": overlap_inner,
        "fraction_genes_overlap_inner": (overlap_inner / n_in) if n_in else np.nan,
        "n_genes_unique_to_dataset": unique_to,
        "n_genes_missing_from_inner": n_in - overlap_inner,
    })
dataset_gene_retention = pd.DataFrame(rows)
dataset_gene_retention.to_csv(GENESET_DIR / "dataset_gene_retention_summary.csv", index=False)
dataset_gene_retention


## 8. 遺伝子存在数の分布を可視化 ＋ inner 整合性チェック

In [ ]:
vc = pd.Series(n_present).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(vc.index.astype(int), vc.to_numpy())
ax.set_xlabel("n_datasets_present")
ax.set_ylabel("genes")
ax.set_title("gene count by number of datasets present")
savefig(GENESET_DIR / "gene_count_by_n_datasets_present.png")

n_in_all = int((n_present == n_datasets).sum())
print(f"genes present in all {n_datasets} datasets: {n_in_all}; inner n_genes: {inner.n_vars}")
if n_in_all != inner.n_vars:
    print(f"[WARN] genes-in-all-datasets ({n_in_all}) != inner gene count ({inner.n_vars})")


## 9. PanglaoDB 参照マーカー遺伝子セットの取得・整形

固定辞書を主ソースにしない。PanglaoDB（decoupler / OmniPath 経由）から取得し、取得できない／
不足する細胞種だけ fallback を使う。最終的に `PANGLAODB_MARKERS` / `marker_source_table` /
`marker_loading_summary` を作る。自動アノテーションは provisional / exploratory。

In [ ]:
PANGLAODB_CELLTYPE_ALIASES = {
    "Motor_Neuron": ["Motor neurons", "Motor neuron", "Neurons"],
    "Neuron": ["Neurons", "Neuron"],
    "Astrocyte": ["Astrocytes", "Astrocyte"],
    "Oligodendrocyte": ["Oligodendrocytes", "Oligodendrocyte"],
    "OPC": ["Oligodendrocyte precursor cells", "Oligodendrocyte precursor cell", "OPCs", "OPC"],
    "Microglia": ["Microglia"],
    "Macrophage": ["Macrophages", "Macrophage"],
    "Endothelial": ["Endothelial cells", "Endothelial cell"],
    "Pericyte": ["Pericytes", "Pericyte"],
    "Ependymal": ["Ependymal cells", "Ependymal cell"],
    "Schwann_Meningeal": ["Schwann cells", "Schwann cell", "Meningeal cells", "Meningeal cell", "Fibroblasts"],
    "Fibroblast": ["Fibroblasts", "Fibroblast"],
    "Immune": ["Immune cells", "Leukocytes"],
    "T_cell": ["T cells", "T cell"],
    "B_cell": ["B cells", "B cell"],
    "NK_cell": ["NK cells", "Natural killer cells", "NK cell"],
}

FALLBACK_MARKERS = {
    "Motor_Neuron": ["Chat", "Slc18a3", "Mnx1", "Isl1", "Isl2", "Chodl"],
    "Neuron": ["Snap25", "Syt1", "Rbfox3", "Tubb3", "Map2"],
    "Astrocyte": ["Aqp4", "Gfap", "Aldh1l1", "Slc1a3", "Slc1a2"],
    "Oligodendrocyte": ["Mog", "Mbp", "Plp1", "Cnp", "Mag", "Mobp"],
    "OPC": ["Pdgfra", "Cspg4", "Olig1", "Olig2", "Sox10"],
    "Microglia": ["P2ry12", "Tmem119", "Cx3cr1", "Hexb", "Aif1", "Sall1", "Csf1r"],
    "Macrophage": ["Lyz2", "Cd68", "Adgre1", "Itgam", "C1qa", "C1qb", "C1qc", "Apoe"],
    "Endothelial": ["Pecam1", "Kdr", "Cldn5", "Flt1", "Vwf", "Tek", "Esam"],
    "Pericyte": ["Pdgfrb", "Rgs5", "Kcnj8", "Acta2", "Des", "Cspg4"],
    "Ependymal": ["Foxj1", "Tmem212", "Pifo", "Dynlrb2", "Rsph1"],
    "Schwann_Meningeal": ["Mpz", "Pmp22", "Sox10", "Dcn", "Col1a1", "Col1a2", "Lum"],
    "Fibroblast": ["Col1a1", "Col1a2", "Dcn", "Lum", "Pdgfra", "Lox"],
    "Immune": ["Ptprc", "Lcp1", "Tyrobp"],
    "T_cell": ["Cd3d", "Cd3e", "Cd2", "Trac", "Lck"],
    "B_cell": ["Ms4a1", "Cd79a", "Cd79b", "Cd19", "Mzb1"],
    "NK_cell": ["Nkg7", "Klrb1c", "Gzma", "Gzmb", "Prf1"],
}

MARKER_CONFIG = {
    "organism": "mouse",
    "min_panglaodb_markers": 3,
    "max_markers_per_cell_type": 50,
    "allow_fallback": True,
}

CELLTYPE_COL_CANDIDATES = ["cell_type", "celltype", "source", "name"]
GENE_COL_CANDIDATES = ["genesymbol", "gene_symbol", "target", "gene", "symbol"]
ORGANISM_COL_CANDIDATES = ["organism", "species"]
CANONICAL_COL_CANDIDATES = ["canonical_marker", "canonical", "is_canonical"]


In [ ]:
def load_panglaodb_marker_table(organism="mouse"):
    """PanglaoDB marker table を decoupler / OmniPath 経由で取得。失敗時は None（落とさない）。"""
    try:
        import decoupler as dc
    except Exception as e:
        print(f"[WARN] decoupler import failed: {e}")
        return None

    getters = []
    getters.append(lambda: dc.get_resource("PanglaoDB", organism=organism))
    getters.append(lambda: dc.get_resource("PanglaoDB"))
    getters.append(lambda: dc.op.get_resource("PanglaoDB", organism=organism))  # newer API
    getters.append(lambda: dc.op.get_resource("PanglaoDB"))
    for g in getters:
        try:
            tbl = g()
            if tbl is not None and len(tbl):
                print(f"[INFO] PanglaoDB table loaded: shape={tbl.shape}")
                return tbl
        except Exception:
            continue
    print("[WARN] could not load PanglaoDB marker table via decoupler/OmniPath; will use fallback")
    return None


def _first_present(df, candidates):
    if df is None:
        return None
    low = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None


panglaodb_table = load_panglaodb_marker_table(MARKER_CONFIG["organism"])
CT_COL = _first_present(panglaodb_table, CELLTYPE_COL_CANDIDATES)
GENE_COL = _first_present(panglaodb_table, GENE_COL_CANDIDATES)
ORG_COL = _first_present(panglaodb_table, ORGANISM_COL_CANDIDATES)
CANON_COL = _first_present(panglaodb_table, CANONICAL_COL_CANDIDATES)
print("detected columns -> celltype:", CT_COL, "gene:", GENE_COL, "organism:", ORG_COL, "canonical:", CANON_COL)


In [ ]:
# outer/inner の var lookup（normalized -> 実際の var_name）
outer_lookup = {normalize_gene_symbol(g): g for g in outer.var_names}
inner_lookup = {normalize_gene_symbol(g): g for g in inner.var_names}

PANGLAODB_MARKERS = {cat: [] for cat in PANGLAODB_CELLTYPE_ALIASES}
marker_source_rows = []
alias_rows = []
loading_rows = []

for cat, aliases in PANGLAODB_CELLTYPE_ALIASES.items():
    pang_norm = []   # normalized gene symbols from PanglaoDB
    if panglaodb_table is not None and CT_COL and GENE_COL:
        sub = panglaodb_table[panglaodb_table[CT_COL].astype(str).isin(aliases)].copy()
        if ORG_COL is not None and len(sub):
            om = sub[ORG_COL].astype(str).str.lower().str.contains(MARKER_CONFIG["organism"].lower())
            if om.any():
                sub = sub[om]
        # canonical 優先（ある場合）
        if CANON_COL is not None and len(sub):
            is_can = sub[CANON_COL].astype(str).str.lower().isin(["true", "1", "yes", "canonical"])
            sub = pd.concat([sub[is_can], sub[~is_can]])
        for alias in aliases:
            arows = sub[sub[CT_COL].astype(str) == alias]
            an = [normalize_gene_symbol(x) for x in arows[GENE_COL].tolist()]
            an = [x for x in an if x]
            alias_rows.append({
                "marker_category": cat,
                "panglaodb_alias": alias,
                "n_rows_matched": int(len(arows)),
                "n_unique_markers_matched": int(len(set(an))),
            })
        seen = set()
        for x in [normalize_gene_symbol(v) for v in sub[GENE_COL].tolist()]:
            if x and x not in seen:
                seen.add(x)
                pang_norm.append(x)
    else:
        for alias in aliases:
            alias_rows.append({"marker_category": cat, "panglaodb_alias": alias,
                               "n_rows_matched": 0, "n_unique_markers_matched": 0})

    # PanglaoDB markers を実際の var 表記へ（outer 優先）
    pang_in_var, pang_norm_kept = [], []
    for x in pang_norm:
        g = outer_lookup.get(x) or inner_lookup.get(x)
        if g is not None and g not in pang_in_var:
            pang_in_var.append(g)
            pang_norm_kept.append(x)
    pang_in_var = pang_in_var[: MARKER_CONFIG["max_markers_per_cell_type"]]
    pang_norm_kept = pang_norm_kept[: MARKER_CONFIG["max_markers_per_cell_type"]]
    n_from_pang = len(pang_in_var)

    final = list(pang_in_var)
    fallback_used = False
    if n_from_pang < MARKER_CONFIG["min_panglaodb_markers"] and MARKER_CONFIG["allow_fallback"]:
        for raw in FALLBACK_MARKERS.get(cat, []):
            x = normalize_gene_symbol(raw)
            g = outer_lookup.get(x) or inner_lookup.get(x)
            if g is not None and g not in final:
                final.append(g)
                fallback_used = True
                marker_source_rows.append({
                    "marker_category": cat, "marker_gene": g, "marker_gene_normalized": x,
                    "source": "Fallback", "panglaodb_cell_type": "", "used_for_scoring": True,
                    "in_outer": x in outer_lookup, "in_inner": x in inner_lookup, "fallback_used": True,
                })
    # PanglaoDB 由来の source rows
    for g, x in zip(pang_in_var, pang_norm_kept):
        marker_source_rows.append({
            "marker_category": cat, "marker_gene": g, "marker_gene_normalized": x,
            "source": "PanglaoDB", "panglaodb_cell_type": ";".join(aliases), "used_for_scoring": True,
            "in_outer": x in outer_lookup, "in_inner": x in inner_lookup, "fallback_used": False,
        })

    PANGLAODB_MARKERS[cat] = final
    n_fb = sum(1 for r in marker_source_rows if r["marker_category"] == cat and r["source"] == "Fallback")
    warn = ""
    if len(final) == 0:
        warn_s = "no markers"
    n_in_outer = sum(1 for g in final if normalize_gene_symbol(g) in outer_lookup)
    n_in_inner = sum(1 for g in final if normalize_gene_symbol(g) in inner_lookup)
    loading_rows.append({
        "marker_category": cat,
        "n_markers_from_panglaodb": n_from_pang,
        "n_markers_from_fallback": n_fb,
        "n_markers_total": len(final),
        "n_markers_in_outer": n_in_outer,
        "n_markers_in_inner": n_in_inner,
        "fallback_used": fallback_used,
        "warning": "fallback_used" if fallback_used else ("no_markers" if not final else ""),
    })

marker_source_table = pd.DataFrame(marker_source_rows)
marker_loading_summary = pd.DataFrame(loading_rows)
alias_map = pd.DataFrame(alias_rows)

marker_source_table.to_csv(ANNOTATION_DIR / "panglaodb_marker_source_table.csv", index=False)
alias_map.to_csv(ANNOTATION_DIR / "panglaodb_celltype_alias_map.csv", index=False)
marker_loading_summary.to_csv(ANNOTATION_DIR / "panglaodb_marker_loading_summary.csv", index=False)
print("PanglaoDB table available:", panglaodb_table is not None)
marker_loading_summary


In [ ]:
# 9-8 warnings
if panglaodb_table is None:
    print("[WARN] PanglaoDB marker table could not be loaded; fallback markers used where available")
for _, r in marker_loading_summary.iterrows():
    cat = r["marker_category"]
    if r["n_markers_from_panglaodb"] == 0:
        print(f"[WARN] {cat}: 0 markers from PanglaoDB")
    if r["n_markers_in_outer"] < 2:
        print(f"[WARN] {cat}: <2 markers present in data (outer={r['n_markers_in_outer']})")
    if r["fallback_used"] and cat in ("Motor_Neuron", "Ependymal", "Schwann_Meningeal"):
        print(f"[WARN] {cat}: fallback used for an unstable-alias cell type")

def _overlap(a, b):
    sa = set(normalize_gene_symbol(x) for x in PANGLAODB_MARKERS.get(a, []))
    sb = set(normalize_gene_symbol(x) for x in PANGLAODB_MARKERS.get(b, []))
    return sa & sb

if len(_overlap("Microglia", "Macrophage")) >= 2:
    print("[WARN] Microglia/Macrophage markers overlap substantially:", sorted(_overlap("Microglia", "Macrophage")))
for a, b in [("OPC", "Oligodendrocyte"), ("OPC", "Schwann_Meningeal"), ("Oligodendrocyte", "Schwann_Meningeal")]:
    ov = _overlap(a, b)
    if ov:
        print(f"[WARN] {a}/{b} share lineage markers:", sorted(ov))
print("done marker warnings")


## 10. 既知マーカー遺伝子の保持確認（outer / inner）

In [ ]:
present_map = gene_presence_summary.set_index("gene")
rows = []
for cat, genes in PANGLAODB_MARKERS.items():
    for g in genes:
        gn = normalize_gene_symbol(g)
        in_outer = gn in outer_lookup
        in_inner = gn in inner_lookup
        og = outer_lookup.get(gn)
        if og is not None and og in present_map.index:
            ndp = int(present_map.loc[og, "n_datasets_present"])
            miss = present_map.loc[og, "datasets_missing"]
            pct = float(present_map.loc[og, "pct_nonzero_cells_outer"])
        else:
            ndp, miss, pct = 0, "", np.nan
        rows.append({
            "marker_gene": g, "marker_category": cat,
            "in_outer": in_outer, "in_inner": in_inner,
            "n_datasets_present": ndp, "datasets_missing": miss,
            "pct_nonzero_cells_outer": pct,
        })
marker_gene_retention = pd.DataFrame(rows)
marker_gene_retention.to_csv(GENESET_DIR / "marker_gene_retention_summary.csv", index=False)

cat_rows = []
for cat, genes in PANGLAODB_MARKERS.items():
    n_def = len(genes)
    n_out = sum(1 for g in genes if normalize_gene_symbol(g) in outer_lookup)
    n_in = sum(1 for g in genes if normalize_gene_symbol(g) in inner_lookup)
    cat_rows.append({
        "marker_category": cat, "n_markers_defined": n_def,
        "n_markers_in_outer": n_out, "n_markers_in_inner": n_in,
        "fraction_markers_in_outer": (n_out / n_def) if n_def else np.nan,
        "fraction_markers_in_inner": (n_in / n_def) if n_def else np.nan,
    })
marker_category_retention = pd.DataFrame(cat_rows)
marker_category_retention.to_csv(GENESET_DIR / "marker_category_retention_summary.csv", index=False)
marker_category_retention


## 11. MT / ribosomal genes の保持確認

In [ ]:
mt_outer = int(get_mt_mask_from_names(outer.var_names).sum())
mt_inner = int(get_mt_mask_from_names(inner.var_names).sum())
ribo_outer = int(get_ribo_mask_from_names(outer.var_names).sum())
ribo_inner = int(get_ribo_mask_from_names(inner.var_names).sum())

mt_ribo = pd.DataFrame([
    {"gene_class": "MT", "n_in_outer": mt_outer, "n_in_inner": mt_inner},
    {"gene_class": "ribosomal", "n_in_outer": ribo_outer, "n_in_inner": ribo_inner},
])
mt_ribo.to_csv(GENESET_DIR / "mt_ribo_gene_retention_summary.csv", index=False)
if mt_inner == 0:
    print("[WARN] no MT genes remain in inner; MT-based QC visualization on inner is unreliable")
elif mt_inner < 5:
    print(f"[WARN] few MT genes in inner ({mt_inner})")
mt_ribo


## 12. gene symbol 不一致の可能性を確認（05 では修正せず warning のみ）

In [ ]:
names = pd.Index(outer.var_names).astype(str)
diag = {
    "n_genes": len(names),
    "n_unique": int(names.nunique()),
    "has_duplicates": bool(names.duplicated().any()),
    "n_duplicates": int(names.duplicated().sum()),
    "any_lowercase": bool(any(n != n.upper() for n in names)),
    "n_ensembl_like": int(names.str.match(r"^ENS[A-Z]*G?\d+").sum()),
    "n_mt_dash_upper": int(names.str.upper().str.startswith("MT-").sum()),
    "n_mt_dash_mixed": int(sum(n.upper().startswith("MT-") and not n.startswith("MT-") for n in names)),
}
gene_symbol_diag = pd.DataFrame({"key": list(diag.keys()), "value": list(diag.values())})
gene_symbol_diag.to_csv(GENESET_DIR / "gene_symbol_diagnostics.csv", index=False)
if diag["has_duplicates"]:
    print(f"[WARN] duplicate gene names: {diag['n_duplicates']}")
if diag["n_ensembl_like"] > 0:
    print(f"[WARN] possible Ensembl IDs mixed in gene names: {diag['n_ensembl_like']}")
gene_symbol_diag


## 13. 探索用 log-expression copy を作る

`make_logexpr_copy_from_original_scale()` は入力を破壊せず copy を返す。探索用であり、scVI 等の
count input には使わない。PCA/UMAP/Harmony/clustering はまず `inner_log` を使う。

In [ ]:
def make_logexpr_copy_from_original_scale(adata, state_col="qc_preprocessing_state", target_sum=1e4):
    """Return a NEW AnnData converted to rough common log space. Does NOT modify input.

    raw_count_like / cpm_tpm_like: normalize_total(target_sum) + log1p
    log_normalized_like: keep as is

    Exploratory / visualization only; NOT a raw-count matrix; do not feed to scVI.
    """
    parts = []
    for state in adata.obs[state_col].astype(str).unique():
        sub = adata[adata.obs[state_col].astype(str) == state].copy()
        if state in ("raw_count_like", "cpm_tpm_like"):
            sc.pp.normalize_total(sub, target_sum=target_sum)
            sc.pp.log1p(sub)
        elif state == "log_normalized_like":
            pass
        else:
            print(f"[WARN] unknown preprocessing state {state}; keeping .X as is")
        parts.append(sub)
    out = ad.concat(parts, join="outer", merge="same", index_unique=None)
    # 元の cell / gene 順に戻す（名前で 2D 選択するので X と var/obs は整合する）
    out = out[adata.obs_names, adata.var_names].copy()
    out.uns["made_from"] = "merged_qc_original_scale"
    out.uns["logexpr_note"] = "exploratory only; not raw counts; not for scVI"
    return out


outer_log = make_logexpr_copy_from_original_scale(outer)
inner_log = make_logexpr_copy_from_original_scale(inner)
# 元 adata は破壊しない（確認）
print("outer.X unchanged check (max):", float(outer.X.max()))
print("outer_log:", outer_log.shape, " inner_log:", inner_log.shape)


## 14. inner / outer-only 遺伝子の発現・検出状況比較

In [ ]:
def gene_metrics(adata_log):
    X = adata_log.X
    if sp.issparse(X):
        nnz = np.asarray(X.getnnz(axis=0)).ravel()
        mean = np.asarray(X.mean(axis=0)).ravel()
        meansq = np.asarray(X.multiply(X).mean(axis=0)).ravel()
    else:
        Xd = np.asarray(X)
        nnz = (Xd != 0).sum(axis=0)
        mean = Xd.mean(axis=0)
        meansq = (Xd ** 2).mean(axis=0)
    var = meansq - mean ** 2
    return nnz, mean, var

# outer 上で HVG を一度計算（sec 14, 15 で共有）
sc.pp.highly_variable_genes(outer_log, n_top_genes=min(3000, outer_log.n_vars - 1), flavor="seurat")

nnz, mean, var = gene_metrics(outer_log)
gm = pd.DataFrame({
    "gene": list(outer_log.var_names),
    "n_nonzero_cells": nnz,
    "pct_nonzero_cells": 100.0 * nnz / outer_log.n_obs,
    "mean_expression": mean,
    "variance": var,
    "is_hvg_outer": outer_log.var["highly_variable"].to_numpy(),
    "dispersions_norm": outer_log.var["dispersions_norm"].to_numpy() if "dispersions_norm" in outer_log.var.columns else np.nan,
})
gm["group"] = np.where(gm["gene"].isin(inner_gene_set), "inner", "outer_only")
gm.to_csv(GENESET_DIR / "inner_vs_outer_only_gene_metrics.csv", index=False)

for col, fname, logx in [
    ("pct_nonzero_cells", "inner_vs_outer_only_pct_nonzero_cells.png", False),
    ("mean_expression", "inner_vs_outer_only_mean_expression.png", False),
    ("variance", "inner_vs_outer_only_variance.png", False),
]:
    fig, ax = plt.subplots(figsize=(6, 4))
    for grp, color in [("inner", "tab:blue"), ("outer_only", "tab:orange")]:
        vals = gm.loc[gm["group"] == grp, col].to_numpy()
        vals = vals[np.isfinite(vals)]
        if vals.size:
            ax.hist(vals, bins=60, alpha=0.5, label=grp, color=color, density=True)
    ax.set_xlabel(col)
    ax.set_ylabel("density")
    ax.set_title(f"{col}: inner vs outer_only")
    ax.legend()
    savefig(GENESET_DIR / fname)
gm.groupby("group")[["pct_nonzero_cells", "mean_expression", "variance"]].median()


## 15. HVG と inner の重なり確認

In [ ]:
hvg_outer_set = set(outer_log.var_names[outer_log.var["highly_variable"]])
n_hvg = len(hvg_outer_set)
n_hvg_in_inner = len(hvg_outer_set & inner_gene_set)
n_hvg_outer_only = n_hvg - n_hvg_in_inner

# inner 内で HVG を選んだ場合
sc.pp.highly_variable_genes(inner_log, n_top_genes=min(3000, inner_log.n_vars - 1), flavor="seurat")
hvg_inner_set = set(inner_log.var_names[inner_log.var["highly_variable"]])
overlap_innerHVG_outerHVG = len(hvg_inner_set & hvg_outer_set)

hvg_overlap_summary = pd.DataFrame([{
    "n_hvg_outer": n_hvg,
    "n_hvg_outer_in_inner": n_hvg_in_inner,
    "n_hvg_outer_only": n_hvg_outer_only,
    "fraction_hvg_outer_in_inner": (n_hvg_in_inner / n_hvg) if n_hvg else np.nan,
    "n_hvg_inner": len(hvg_inner_set),
    "n_overlap_innerHVG_outerHVG": overlap_innerHVG_outerHVG,
}])
hvg_overlap_summary.to_csv(GENESET_DIR / "hvg_inner_overlap_summary.csv", index=False)

hvg_gene_table = pd.DataFrame({"gene": sorted(hvg_outer_set)})
hvg_gene_table["in_inner"] = hvg_gene_table["gene"].isin(inner_gene_set)
hvg_gene_table["in_inner_hvg"] = hvg_gene_table["gene"].isin(hvg_inner_set)
hvg_gene_table.to_csv(GENESET_DIR / "hvg_gene_table.csv", index=False)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["in_inner", "outer_only"], [n_hvg_in_inner, n_hvg_outer_only], color=["tab:blue", "tab:orange"])
ax.set_ylabel("n outer HVGs")
ax.set_title("outer HVG: inner vs outer_only")
savefig(GENESET_DIR / "hvg_inner_overlap_barplot.png")

fig, ax = plt.subplots(figsize=(6, 4))
disp = outer_log.var.loc[outer_log.var["highly_variable"], "dispersions_norm"] if "dispersions_norm" in outer_log.var.columns else None
if disp is not None:
    in_in = [g in inner_gene_set for g in disp.index]
    ax.hist(disp[in_in].to_numpy(), bins=50, alpha=0.5, label="inner", density=True)
    ax.hist(disp[[not x for x in in_in]].to_numpy(), bins=50, alpha=0.5, label="outer_only", density=True)
    ax.set_xlabel("dispersions_norm (outer HVG)")
    ax.legend()
ax.set_title("HVG rank/dispersion: inner vs outer_only")
savefig(GENESET_DIR / "hvg_rank_inner_vs_outer_only.png")
hvg_overlap_summary


## 16. inner gene set を使ってよいかの判断材料

In [ ]:
min_frac_overlap = float(dataset_gene_retention["fraction_genes_overlap_inner"].min()) if len(dataset_gene_retention) else np.nan
mean_marker_frac_inner = float(marker_category_retention["fraction_markers_in_inner"].mean())
hvg_in_inner_frac = float(hvg_overlap_summary["fraction_hvg_outer_in_inner"].iloc[0])

decision_rows = [
    {"metric": "inner_n_genes", "value": inner.n_vars, "note": "inner 遺伝子数"},
    {"metric": "fraction_hvg_outer_in_inner", "value": round(hvg_in_inner_frac, 4), "note": "outer HVG のうち inner に残る割合（高いほど良い）"},
    {"metric": "mean_fraction_markers_in_inner", "value": round(mean_marker_frac_inner, 4), "note": "細胞種マーカーの inner 保持率（高いほど良い）"},
    {"metric": "min_dataset_fraction_genes_overlap_inner", "value": round(min_frac_overlap, 4) if min_frac_overlap == min_frac_overlap else np.nan, "note": "最も inner を制限している dataset の重なり率"},
    {"metric": "mt_genes_in_inner", "value": mt_inner, "note": "inner の MT 遺伝子数"},
    {"metric": "gene_name_has_duplicates", "value": diag["has_duplicates"], "note": "gene 名重複"},
    {"metric": "gene_name_ensembl_like", "value": diag["n_ensembl_like"], "note": "Ensembl ID 混在の疑い"},
]
verdict = "inner_looks_ok"
if hvg_in_inner_frac < 0.5 or mean_marker_frac_inner < 0.5 or inner.n_vars < 2000:
    verdict = "inner_may_be_insufficient_consider_intermediate_gene_set"
decision_rows.append({"metric": "verdict", "value": verdict, "note": "目安。最終判断は人間が行う"})
inner_decision = pd.DataFrame(decision_rows)
inner_decision.to_csv(GENESET_DIR / "inner_gene_set_decision_summary.csv", index=False)

# NOTE: inner が不十分そうな場合の中間 gene set 案（コメント。デフォルトでは使わない）:
# min_datasets = max(2, int(0.8 * n_datasets))
# intermediate_genes = gene_presence_summary.loc[gene_presence_summary["n_datasets_present"] >= min_datasets, "gene"].tolist()
# adata_inter = outer_log[:, [g for g in intermediate_genes if g in outer_log.var_names]].copy()

inner_decision


## 17. PCA / UMAP before Harmony（inner_log を使用）

細胞数が多く重い場合は可視化用サンプリング可（その場合ファイル名/Markdown に `sampled` を明記）。
ここではデフォルトで全細胞を使う。

In [ ]:
adata_pca = inner_log[:, inner_log.var["highly_variable"]].copy()
n_comps = int(min(50, adata_pca.n_vars - 1, adata_pca.n_obs - 1))
sc.pp.pca(adata_pca, n_comps=n_comps)
sc.pp.neighbors(adata_pca, use_rep="X_pca")
sc.tl.umap(adata_pca)
adata_pca.obsm["X_umap_before_harmony"] = adata_pca.obsm["X_umap"].copy()

BEFORE_KEYS = ["dataset_id", "Condition", "qc_preprocessing_state",
               "pct_mt_for_qc", "n_nonzero_genes", "n_nonzero_mt_genes"]
emb_color(adata_pca, "umap_before_harmony", BEFORE_KEYS, "umap_before_harmony", PCA_UMAP_DIR)
emb_color(adata_pca, "pca", ["dataset_id", "Condition", "qc_preprocessing_state"], "pca_before_harmony", PCA_UMAP_DIR)
print("PCA/UMAP before Harmony done; n_comps =", n_comps)


## 18. Clustering before Harmony（Leiden、無ければ Louvain、両方不可ならskip）

In [ ]:
def run_clustering(adata, resolution, key_added):
    try:
        sc.tl.leiden(adata, resolution=resolution, key_added=key_added)
        return True
    except Exception as e1:
        print(f"  [warn] leiden failed ({e1}); trying louvain")
        try:
            sc.tl.louvain(adata, resolution=resolution, key_added=key_added)
            return True
        except Exception as e2:
            print(f"  [warn] louvain also failed ({e2}); skip {key_added}")
            return False


def cluster_reports(adata, cluster_key, tag, outdir):
    if cluster_key not in adata.obs.columns:
        print(f"  [skip] {cluster_key} not present")
        return
    obs = adata.obs
    obs[cluster_key].value_counts().sort_index().to_csv(outdir / f"cluster_{tag}_cell_counts.csv", header=["n_cells"])
    for gkey, name in [("dataset_id", "dataset"), ("Condition", "condition")]:
        if gkey not in obs.columns:
            continue
        ct = obs.groupby([cluster_key, gkey], observed=True).size().unstack(fill_value=0)
        ct.to_csv(outdir / f"cluster_{tag}_{name}_composition.csv")
        stacked_fraction_plot(ct, f"cluster x {name} fraction ({tag})",
                              outdir / f"cluster_{tag}_{name}_fraction.png", legend_title=name)


ok_b05 = run_clustering(adata_pca, 0.5, "leiden_before_harmony_r05")
ok_b10 = run_clustering(adata_pca, 1.0, "leiden_before_harmony_r10")
for k, fn in [("leiden_before_harmony_r05", "umap_before_harmony_leiden_r05.png"),
              ("leiden_before_harmony_r10", "umap_before_harmony_leiden_r10.png")]:
    if k in adata_pca.obs.columns:
        try:
            sc.pl.embedding(adata_pca, basis="umap_before_harmony", color=k, show=False)
            savefig(CLUSTER_DIR / fn)
        except Exception as e:
            print(f"  [warn] {fn}: {e}"); plt.close("all")
if ok_b05:
    cluster_reports(adata_pca, "leiden_before_harmony_r05", "before_harmony", CLUSTER_DIR)
print("before-harmony clustering done")


## 19. Harmony（batch key = dataset_id、使えなければ skip）

In [ ]:
HARMONY_OK = False
try:
    import scanpy.external as sce
    sce.pp.harmony_integrate(adata_pca, key="dataset_id")
    HARMONY_OK = "X_pca_harmony" in adata_pca.obsm
    print("Harmony done:", HARMONY_OK)
except Exception as e:
    print(f"[WARN] Harmony skipped: {e}")


## 20. UMAP after Harmony

In [ ]:
if HARMONY_OK:
    sc.pp.neighbors(adata_pca, use_rep="X_pca_harmony")
    sc.tl.umap(adata_pca)
    adata_pca.obsm["X_umap_after_harmony"] = adata_pca.obsm["X_umap"].copy()
    AFTER_KEYS = ["dataset_id", "Condition", "qc_preprocessing_state",
                  "pct_mt_for_qc", "n_nonzero_genes", "n_nonzero_mt_genes"]
    emb_color(adata_pca, "umap_after_harmony", AFTER_KEYS, "umap_after_harmony", PCA_UMAP_DIR)
    print("UMAP after Harmony done")
else:
    print("[skip] Harmony not available; after-harmony UMAP skipped")


## 21. Clustering after Harmony

In [ ]:
if HARMONY_OK:
    run_clustering(adata_pca, 0.5, "leiden_harmony_r05")
    run_clustering(adata_pca, 1.0, "leiden_harmony_r10")
    for k, fn in [("leiden_harmony_r05", "umap_after_harmony_leiden_r05.png"),
                  ("leiden_harmony_r10", "umap_after_harmony_leiden_r10.png")]:
        if k in adata_pca.obs.columns:
            try:
                sc.pl.embedding(adata_pca, basis="umap_after_harmony", color=k, show=False)
                savefig(CLUSTER_DIR / fn)
            except Exception as e:
                print(f"  [warn] {fn}: {e}"); plt.close("all")
    if "leiden_harmony_r05" in adata_pca.obs.columns:
        cluster_reports(adata_pca, "leiden_harmony_r05", "after_harmony", CLUSTER_DIR)
    print("after-harmony clustering done")
else:
    print("[skip] after-harmony clustering skipped")

# 後続で使う主クラスタ列とアノテーション用 UMAP basis
if HARMONY_OK and "leiden_harmony_r05" in adata_pca.obs.columns:
    CLUSTER_KEY = "leiden_harmony_r05"
    ANNO_BASIS = "umap_after_harmony"
elif "leiden_before_harmony_r05" in adata_pca.obs.columns:
    CLUSTER_KEY = "leiden_before_harmony_r05"
    ANNO_BASIS = "umap_before_harmony"
else:
    CLUSTER_KEY = None
    ANNO_BASIS = "umap_before_harmony"
print("CLUSTER_KEY:", CLUSTER_KEY, " ANNO_BASIS:", ANNO_BASIS)


## 22. known annotation で UMAP を色付け（before / after Harmony）

In [ ]:
KNOWN_KEYS = [
    "sex", "cell_type", "meta_sex", "meta_cell.type.refined",
    "meta_microglia.subclusters", "meta_provisional.microglia.subclusters",
    "meta_astrocyte.subclusters", "meta_oligodendrocyte.subclusters",
    "Age", "cell_type_original_GSE206330",
]
emb_color(adata_pca, "umap_before_harmony", KNOWN_KEYS, "umap_before_harmony", PCA_UMAP_DIR)
if HARMONY_OK:
    emb_color(adata_pca, "umap_after_harmony", KNOWN_KEYS, "umap_after_harmony", PCA_UMAP_DIR)
print("known annotation coloring done")


## 23. marker score によるクラスタ自動アノテーション（provisional / exploratory）

`PANGLAODB_MARKERS` を `sc.tl.score_genes` でスコア化し、クラスタ平均から top1 cell type を割り当てる。
**これは provisional annotation であり本番ではない。**

In [ ]:
AUTO_ANNOTATION = {"min_score_delta": 0.05, "min_markers_used": 2}

usage_rows = []
for cat, genes in PANGLAODB_MARKERS.items():
    used = [g for g in genes if g in adata_pca.var_names]
    missing = [g for g in genes if g not in adata_pca.var_names]
    if used:
        sc.tl.score_genes(adata_pca, used, score_name=f"score_marker_{cat}")
    else:
        adata_pca.obs[f"score_marker_{cat}"] = np.nan
    usage_rows.append({
        "cell_type": cat, "n_markers_defined": len(genes), "n_markers_used": len(used),
        "markers_used": ";".join(used), "markers_missing": ";".join(missing),
    })
marker_score_gene_usage = pd.DataFrame(usage_rows)
marker_score_gene_usage.to_csv(ANNOTATION_DIR / "marker_score_gene_usage.csv", index=False)
markers_used_count = {r["cell_type"]: r["n_markers_used"] for r in usage_rows}
marker_score_gene_usage


In [ ]:
if CLUSTER_KEY is None:
    print("[skip] no cluster key; auto annotation skipped")
    cluster_marker_score = pd.DataFrame()
    cluster_auto = pd.DataFrame()
else:
    score_cols = [c for c in adata_pca.obs.columns if c.startswith("score_marker_")]
    cluster_marker_score = adata_pca.obs.groupby(CLUSTER_KEY, observed=True)[score_cols].mean()
    cluster_marker_score.to_csv(ANNOTATION_DIR / "cluster_marker_score_matrix.csv")

    def dominant(series):
        vc = series.astype(str).value_counts()
        return vc.index[0] if len(vc) else "NA"

    rows = []
    cell_auto = {}
    for cl, row in cluster_marker_score.iterrows():
        s = row.dropna().sort_values(ascending=False)
        cells_in = adata_pca.obs[adata_pca.obs[CLUSTER_KEY] == cl]
        top1 = s.index[0].replace("score_marker_", "") if len(s) else "NA"
        top1s = float(s.iloc[0]) if len(s) else np.nan
        top2 = s.index[1].replace("score_marker_", "") if len(s) > 1 else "NA"
        top2s = float(s.iloc[1]) if len(s) > 1 else np.nan
        delta = (top1s - top2s) if (len(s) > 1) else np.nan
        label = top1
        if (markers_used_count.get(top1, 0) < AUTO_ANNOTATION["min_markers_used"]) or \
           (delta == delta and delta < AUTO_ANNOTATION["min_score_delta"]):
            label = f"ambiguous_{top1}"
        cell_auto[cl] = (label, top1s, delta)
        rows.append({
            "cluster": cl, "n_cells": int(cells_in.shape[0]),
            "top1_cell_type": top1, "top1_score": top1s,
            "top2_cell_type": top2, "top2_score": top2s, "score_delta": delta,
            "auto_label": label,
            "dominant_known_cell_type": dominant(cells_in["cell_type"]) if "cell_type" in cells_in.columns else "NA",
            "dominant_dataset_id": dominant(cells_in["dataset_id"]) if "dataset_id" in cells_in.columns else "NA",
            "dominant_Condition": dominant(cells_in["Condition"]) if "Condition" in cells_in.columns else "NA",
        })
    cluster_auto = pd.DataFrame(rows)
    cluster_auto.to_csv(ANNOTATION_DIR / "cluster_auto_annotation_summary.csv", index=False)

    cl_series = adata_pca.obs[CLUSTER_KEY].astype(str)
    adata_pca.obs["auto_cell_type_marker"] = cl_series.map({str(k): v[0] for k, v in cell_auto.items()}).astype(str)
    adata_pca.obs["auto_cell_type_score"] = cl_series.map({str(k): v[1] for k, v in cell_auto.items()}).astype(float)
    adata_pca.obs["auto_cell_type_score_delta"] = cl_series.map({str(k): v[2] for k, v in cell_auto.items()}).astype(float)
    print("auto annotation assigned")
cluster_auto.head() if len(cluster_auto) else "no clusters"


In [ ]:
# 23-4 可視化
if "auto_cell_type_marker" in adata_pca.obs.columns:
    for k, fn in [("auto_cell_type_marker", "umap_auto_cell_type_marker.png"),
                  ("auto_cell_type_score_delta", "umap_auto_cell_type_score_delta.png")]:
        try:
            sc.pl.embedding(adata_pca, basis=ANNO_BASIS, color=k, show=False)
            savefig(ANNOTATION_DIR / fn)
        except Exception as e:
            print(f"  [warn] {fn}: {e}"); plt.close("all")
    if CLUSTER_KEY is not None:
        try:
            sc.pl.embedding(adata_pca, basis=ANNO_BASIS, color=[CLUSTER_KEY, "auto_cell_type_marker"], show=False)
            savefig(ANNOTATION_DIR / "umap_leiden_harmony_r05_with_auto_annotation.png")
        except Exception as e:
            print(f"  [warn] combined auto annotation umap: {e}"); plt.close("all")

# 23-5 dotplot / matrixplot（各 cell type 上位 3-5 個の存在 marker）
if CLUSTER_KEY is not None:
    dot_markers = {}
    for cat, genes in PANGLAODB_MARKERS.items():
        present = [g for g in genes if g in adata_pca.var_names][:5]
        if present:
            dot_markers[cat] = present
    if dot_markers:
        try:
            sc.pl.dotplot(adata_pca, dot_markers, groupby=CLUSTER_KEY, show=False)
            savefig(ANNOTATION_DIR / "marker_dotplot_by_cluster.png")
        except Exception as e:
            print(f"  [warn] dotplot: {e}"); plt.close("all")
        try:
            sc.pl.matrixplot(adata_pca, dot_markers, groupby=CLUSTER_KEY, show=False)
            savefig(ANNOTATION_DIR / "marker_matrixplot_by_cluster.png")
        except Exception as e:
            print(f"  [warn] matrixplot: {e}"); plt.close("all")
print("auto annotation visualization done")


## 24. cluster marker genes（rank_genes_groups）

In [ ]:
if CLUSTER_KEY is not None:
    try:
        sc.tl.rank_genes_groups(adata_pca, groupby=CLUSTER_KEY, method="wilcoxon")
        rgg = sc.get.rank_genes_groups_df(adata_pca, group=None)
        rgg.to_csv(ANNOTATION_DIR / "rank_genes_groups_by_cluster.csv", index=False)
        try:
            sc.pl.rank_genes_groups(adata_pca, n_genes=15, sharey=False, show=False)
            savefig(ANNOTATION_DIR / "rank_genes_groups_by_cluster.png")
        except Exception as e:
            print(f"  [warn] rank_genes plot: {e}"); plt.close("all")
        print("rank_genes_groups done")
    except Exception as e:
        print(f"[WARN] rank_genes_groups failed: {e}")
else:
    print("[skip] no cluster key")


## 25. 自動 annotation と既知 annotation の対応（cross-tab）

In [ ]:
def save_crosstab(auto_col, known_col, path):
    if auto_col not in adata_pca.obs.columns or known_col not in adata_pca.obs.columns:
        print(f"  [skip] crosstab {auto_col} vs {known_col} (missing)")
        return
    obs = adata_pca.obs
    mask = obs[known_col].astype(str).str.lower().isin(["unknown", "nan", "none", ""]) == False
    sub = obs[mask]
    if sub.shape[0] == 0:
        print(f"  [skip] no known cells for {known_col}")
        return
    ct = pd.crosstab(sub[auto_col].astype(str), sub[known_col].astype(str))
    ct.to_csv(path)
    ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0).to_csv(str(path).replace(".csv", "_rownorm.csv"))
    ct.div(ct.sum(axis=0).replace(0, np.nan), axis=1).to_csv(str(path).replace(".csv", "_colnorm.csv"))

save_crosstab("auto_cell_type_marker", "cell_type", ANNOTATION_DIR / "auto_annotation_vs_known_cell_type.csv")
save_crosstab("auto_cell_type_marker", "meta_cell.type.refined", ANNOTATION_DIR / "auto_annotation_vs_meta_cell_type_refined.csv")
save_crosstab("auto_cell_type_marker", "cell_type_original_GSE206330", ANNOTATION_DIR / "auto_annotation_vs_GSE206330_cell_type.csv")
print("crosstabs done")


## 26. Condition / dataset / QC / annotation / cluster をまとめて確認

In [ ]:
COMBINED_KEYS = [
    "source_accession", "dataset_id", "Condition", "qc_preprocessing_state",
    "pct_mt_for_qc", "n_nonzero_genes", "n_nonzero_mt_genes",
    "sex", "cell_type",
    "meta_microglia.subclusters", "meta_astrocyte.subclusters", "meta_oligodendrocyte.subclusters",
    "cell_type_original_GSE206330",
    "leiden_before_harmony_r05", "leiden_harmony_r05", "auto_cell_type_marker",
]
emb_color(adata_pca, "umap_before_harmony", COMBINED_KEYS, "combined_before_harmony", PLOT_DIR)
if HARMONY_OK:
    emb_color(adata_pca, "umap_after_harmony", COMBINED_KEYS, "combined_after_harmony", PLOT_DIR)
print("combined confirmation plots done")


## 27. 探索用 AnnData の保存（任意・デフォルトでは実行しない）

In [ ]:
# 探索用 adata は基本的に保存しない。再実行が重い場合のみ、別名で保存する（original-scale を上書きしない）。
# adata_pca.write_h5ad(OUT_DIR / "inner_logexpr_hvg_pca_umap_harmony_cluster_annotation_check.h5ad")
print("done. exploratory adata not saved by default.")


## まとめ

- このノートブックは merged h5ad の **検収 + 探索的可視化** であり、本解析ではない。
- original-scale の `merged_qc_original_scale_*` は上書きしていない。log-expression 変換は copy に対してのみ実行。
- PanglaoDB ベースの `auto_cell_type_marker` は **provisional / exploratory**。既知 annotation・cluster marker
  genes・marker expression・UMAP 位置・dataset 構成と必ず照合して解釈する。
- inner / outer gene set の妥当性は `gene_set/` 以下の CSV・plot と `inner_gene_set_decision_summary.csv` を参照。
